<h2 style="color:DodgerBlue">Спринт ревью #5</h2> 


<h2 style="color:DodgerBlue">Тема: Разработка информационной системы для учета заказов в компании. </h2> 

Задание: Необходимо разработать консольное приложение на языке программирования (например, C#), которое моделирует работу компании по учету заказов. Программа должна использовать классы для описания сущностей, их атрибутов и методов. Для реализации логики будут использоваться коллекции для хранения связанных данных, а также делегаты и события для автоматизации уведомлений о ключевых действиях в системе.

In [12]:
using System;
using System.Collections.Generic;

public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}

public delegate void OrderStatusChangedHandler(Order order, OrderStatus newStatus);

public class Employee
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string Position { get; set; }
    public int OrdersProcessed { get; set; }

    public Employee(string firstName, string lastName, string position)
    {
        FirstName = firstName;
        LastName = lastName;
        Position = position;
        OrdersProcessed = 0;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public void IncrementProcessed()
    {
        OrdersProcessed++;
    }

    public override string ToString()
    {
        return $"{GetFullName()} — {Position} — Обработано заказов: {OrdersProcessed}";
    }
}

public class Customer
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string PhoneNumber { get; set; }

    public Customer(string firstName, string lastName, string phoneNumber)
    {
        FirstName = firstName;
        LastName = lastName;
        PhoneNumber = phoneNumber;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public override string ToString()
    {
        return $"{GetFullName()} — Телефон: {PhoneNumber}";
    }
}

public class Order
{
    private static int _lastOrderId = 0;

    public int OrderId { get; private set; }
    public string Description { get; set; }
    public Employee AssignedEmployee { get; set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }
    public Customer Customer { get; set; }

    public event OrderStatusChangedHandler StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = ++_lastOrderId;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        if (Status == newStatus)
            return;

        Status = newStatus;

        StatusChanged?.Invoke(this, newStatus);

        if (newStatus == OrderStatus.Completed && AssignedEmployee != null)
        {
            AssignedEmployee.IncrementProcessed();
        }
    }

    public void ShowInfo()
    {
        string employeeName = AssignedEmployee != null ? AssignedEmployee.GetFullName() : "Не назначен";
        Console.WriteLine($"Заказ #{OrderId}: {Description} | Заказчик: {Customer.GetFullName()} | Ответственный: {employeeName} | Статус: {GetStatusText(Status)} | Создан: {CreatedAt:dd.MM.yyyy HH:mm}");
    }

    public static string GetStatusText(OrderStatus status)
    {
        switch (status)
        {
            case OrderStatus.Created: return "Создан";
            case OrderStatus.InProgress: return "В работе";
            case OrderStatus.Completed: return "Завершен";
            default: return status.ToString();
        }
    }
}

public class Company
{
    public string Name { get; set; }
    public List<Employee> Employees { get; set; }
    public List<Order> Orders { get; set; }
    public List<Customer> Customers { get; set; }

    public Company(string name)
    {
        Name = name;
        Employees = new List<Employee>();
        Orders = new List<Order>();
        Customers = new List<Customer>();
    }

    public void AddEmployee(Employee employee)
    {
        Employees.Add(employee);
        Console.WriteLine($"Сотрудник {employee.GetFullName()} добавлен в компанию «{Name}».");
    }

    public void AddCustomer(Customer customer)
    {
        Customers.Add(customer);
        Console.WriteLine($"Заказчик {customer.GetFullName()} добавлен в компанию «{Name}».");
    }

    public Order AddOrder(string description, Customer customer)
    {
        Order order = new Order(description, customer);
        Orders.Add(order);
        Console.WriteLine($"Заказ #{order.OrderId} добавлен в компанию «{Name}».");
        return order;
    }

    public void AssignEmployee(Order order, Employee employee)
    {
        order.AssignedEmployee = employee;
        Console.WriteLine($"Сотрудник {employee.GetFullName()} назначен ответственным за заказ #{order.OrderId}.");
    }

    public void ChangeOrderStatus(Order order, OrderStatus newStatus)
    {
        order.ChangeStatus(newStatus);
    }

    public List<Order> GetOrdersByStatus(OrderStatus status)
    {
        List<Order> result = new List<Order>();
        foreach (var order in Orders)
        {
            if (order.Status == status)
                result.Add(order);
        }
        return result;
    }

    public List<Order> GetOrdersByCustomer(string customerLastName)
    {
        List<Order> result = new List<Order>();
        foreach (var order in Orders)
        {
            if (order.Customer.LastName == customerLastName)
                result.Add(order);
        }
        return result;
    }

    public void GetEmployeeReport()
    {
        Console.WriteLine($"=== Отчёт по сотрудникам компании «{Name}» ===");
        foreach (var employee in Employees)
        {
            Console.WriteLine(employee.ToString());
        }
    }
}

Company company = new Company("Альфа");

Employee employee1 = new Employee("Алексей", "Кузнецов", "Руководитель проектов");
Employee employee2 = new Employee("Мария", "Соколова", "Ведущий разработчик");
Employee employee3 = new Employee("Павел", "Морозов", "QA-инженер");
company.AddEmployee(employee1);
company.AddEmployee(employee2);
company.AddEmployee(employee3);
Console.WriteLine();

Customer customer1 = new Customer("Ольга", "Лебедева", "89001112233");
Customer customer2 = new Customer("Никита", "Орлов", "89004445566");
company.AddCustomer(customer1);
company.AddCustomer(customer2);
Console.WriteLine();

Order order1 = company.AddOrder("Разработка корпоративного сайта", customer1);
Order order2 = company.AddOrder("Создание мобильного приложения", customer2);
Order order3 = company.AddOrder("Тестирование корпоративного сайта", customer1);
Order order4 = company.AddOrder("Настройка облачной инфраструктуры", customer2);
Console.WriteLine();

company.AssignEmployee(order1, employee2);
company.AssignEmployee(order2, employee1);
company.AssignEmployee(order3, employee3);
company.AssignEmployee(order4, employee3);

order1.StatusChanged += OnOrderStatusChanged;
order2.StatusChanged += OnOrderStatusChanged;
order3.StatusChanged += OnOrderStatusChanged;
order4.StatusChanged += OnOrderStatusChanged;

static void OnOrderStatusChanged(Order order, OrderStatus newStatus)
{
    Console.WriteLine($"Заказ №{order.OrderId} изменил статус на: {Order.GetStatusText(newStatus)}");
    if (order.AssignedEmployee != null)
    {
        Console.WriteLine($"[УВЕДОМЛЕНИЕ] Сотрудник {order.AssignedEmployee.GetFullName()} уведомлён об изменении статуса заказа №{order.OrderId}.");
    }
}

Console.WriteLine();
Console.WriteLine("--- Смена статусов ---");
company.ChangeOrderStatus(order1, OrderStatus.InProgress);
company.ChangeOrderStatus(order2, OrderStatus.InProgress);
company.ChangeOrderStatus(order1, OrderStatus.Completed);
company.ChangeOrderStatus(order3, OrderStatus.InProgress);
company.ChangeOrderStatus(order3, OrderStatus.Completed);
company.ChangeOrderStatus(order4, OrderStatus.InProgress);
company.ChangeOrderStatus(order4, OrderStatus.Completed);

Console.WriteLine();
Console.WriteLine("--- Все заказы ---");
foreach (var order in company.Orders)
{
    order.ShowInfo();
}

Console.WriteLine();
Console.WriteLine("--- Заказы в статусе: В работе ---");
var inProgress = company.GetOrdersByStatus(OrderStatus.InProgress);
if (inProgress.Count == 0)
{
    Console.WriteLine("Нет заказов в работе");
}
else
{
    foreach (var order in inProgress)
    {
        order.ShowInfo();
    }
}

Console.WriteLine();
Console.WriteLine("--- Заказы в статусе: Завершен ---");
var completed = company.GetOrdersByStatus(OrderStatus.Completed);
if (completed.Count == 0)
{
    Console.WriteLine("Нет завершённых заказов");
}
else
{
    foreach (var order in completed)
    {
        order.ShowInfo();
    }
}

Console.WriteLine();
Console.WriteLine("--- Поиск заказов по заказчику ---");
var customerOrders = company.GetOrdersByCustomer("Лебедева");
if (customerOrders.Count == 0)
{
    Console.WriteLine("Заказов для заказчика не найдено");
}
else
{
    Console.WriteLine($"Найдено заказов: {customerOrders.Count}");
    foreach (var order in customerOrders)
    {
        order.ShowInfo();
    }
}

company.GetEmployeeReport();

Сотрудник Алексей Кузнецов добавлен в компанию «Альфа».
Сотрудник Мария Соколова добавлен в компанию «Альфа».
Сотрудник Павел Морозов добавлен в компанию «Альфа».

Заказчик Ольга Лебедева добавлен в компанию «Альфа».
Заказчик Никита Орлов добавлен в компанию «Альфа».

Заказ #1 добавлен в компанию «Альфа».
Заказ #2 добавлен в компанию «Альфа».
Заказ #3 добавлен в компанию «Альфа».
Заказ #4 добавлен в компанию «Альфа».

Сотрудник Мария Соколова назначен ответственным за заказ #1.
Сотрудник Алексей Кузнецов назначен ответственным за заказ #2.
Сотрудник Павел Морозов назначен ответственным за заказ #3.
Сотрудник Павел Морозов назначен ответственным за заказ #4.

--- Смена статусов ---
Заказ №1 изменил статус на: В работе
[УВЕДОМЛЕНИЕ] Сотрудник Мария Соколова уведомлён об изменении статуса заказа №1.
Заказ №2 изменил статус на: В работе
[УВЕДОМЛЕНИЕ] Сотрудник Алексей Кузнецов уведомлён об изменении статуса заказа №2.
Заказ №1 изменил статус на: Завершен
[УВЕДОМЛЕНИЕ] Сотрудник Мария Сокол